<a href="https://colab.research.google.com/github/amandarcp78-sudo/ORES5160-2026-x2/blob/main/Synthea_A_Alberino.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import zipfile
import os

zip_path = '/content/synthea_sample_data_csv_apr2020.zip'
extract_dir = '/content/synthea_data'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("Extracted files:")
display(os.listdir(extract_dir))

Extracted files:


['csv']

In [8]:
import pandas as pd
import os

# Dynamically find where the CSV files are located in the extracted directory
csv_dir = extract_dir
for root, dirs, files in os.walk(extract_dir):
    if 'patients.csv' in files:
        csv_dir = root
        break

print(f"Loading CSVs from: {csv_dir}")

# Load all potentially required tables
patients = pd.read_csv(f'{csv_dir}/patients.csv')
encounters = pd.read_csv(f'{csv_dir}/encounters.csv')
organizations = pd.read_csv(f'{csv_dir}/organizations.csv')
providers = pd.read_csv(f'{csv_dir}/providers.csv')
payers = pd.read_csv(f'{csv_dir}/payers.csv')
payer_transitions = pd.read_csv(f'{csv_dir}/payer_transitions.csv')

# Display shape and columns to confirm correct loads
print(f"Patients: {patients.shape}")
print(f"Encounters: {encounters.shape}")
print(f"Organizations: {organizations.shape}")
print(f"Providers: {providers.shape}")
print(f"Payers: {payers.shape}")
print(f"Payer Transitions: {payer_transitions.shape}")

Loading CSVs from: /content/synthea_data/csv
Patients: (1171, 25)
Encounters: (53346, 15)
Organizations: (1119, 11)
Providers: (5855, 12)
Payers: (10, 21)
Payer Transitions: (3801, 5)


In [9]:
# 1. Denormalize encounters: patients, organizations, providers, and payers

# Patients join on patients.Id == encounters.PATIENT
encounters_denorm = encounters.merge(patients, left_on='PATIENT', right_on='Id', how='left', suffixes=('', '_patient'))

# Organizations join on organizations.Id == encounters_denorm.ORGANIZATION
encounters_denorm = encounters_denorm.merge(organizations, left_on='ORGANIZATION', right_on='Id', how='left', suffixes=('', '_organization'))

# Providers join on providers.Id == encounters_denorm.PROVIDER
encounters_denorm = encounters_denorm.merge(providers, left_on='PROVIDER', right_on='Id', how='left', suffixes=('', '_provider'))

# Payers join on payers.Id == encounters_denorm.PAYER
encounters_denorm = encounters_denorm.merge(payers, left_on='PAYER', right_on='Id', how='left', suffixes=('', '_payer'))

print("Denormalized Encounters shape:", encounters_denorm.shape)
display(encounters_denorm.head())

Denormalized Encounters shape: (53346, 84)


,Id,START,STOP,PATIENT,ORGANIZATION,PROVIDER,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,...,UNCOVERED_ENCOUNTERS,COVERED_MEDICATIONS,UNCOVERED_MEDICATIONS,COVERED_PROCEDURES,UNCOVERED_PROCEDURES,COVERED_IMMUNIZATIONS,UNCOVERED_IMMUNIZATIONS,UNIQUE_CUSTOMERS,QOLS_AVG,MEMBER_MONTHS
0,d0c40d10-8d87-447e-836e-99d26ad52ea5,2010-01-23T17:45:28Z,2010-01-23T18:10:28Z,034e9e3b-2def-4559-bb2a-7850888ae060,e002090d-4e92-300e-b41e-7d1f21dee4c6,e6283e46-fd81-3611-9459-0edb1c3da357,6e2f1a2d-27bd-3701-8d08-dae202c58632,ambulatory,185345009,Encounter for symptom,...,0,6159,0,15312,0,7520,0,307,0.907256,78768
1,e88bc3a9-007c-405e-aabc-792a38f4aa2b,2012-01-23T17:45:28Z,2012-01-23T18:00:28Z,034e9e3b-2def-4559-bb2a-7850888ae060,772ee193-bb9f-30eb-9939-21e86c8e4da5,6f1d59a7-a5bd-3cf9-9671-5bad2f351c28,6e2f1a2d-27bd-3701-8d08-dae202c58632,wellness,162673000,General examination of patient (procedure),...,0,6159,0,15312,0,7520,0,307,0.907256,78768
2,8f104aa7-4ca9-4473-885a-bba2437df588,2001-05-01T15:02:18Z,2001-05-01T15:17:18Z,1d604da9-9a81-4ba9-80c2-de3375d59b40,5d4b9df1-93ae-3bc9-b680-03249990e558,af01a385-31d3-3c77-8fdb-2867fe88df2f,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,ambulatory,185345009,Encounter for symptom,...,10116,0,5465,0,8281,0,3644,99,1.134518,35196
3,b85c339a-6076-43ed-b9d0-9cf013dec49d,2011-07-28T15:02:18Z,2011-07-28T15:17:18Z,1d604da9-9a81-4ba9-80c2-de3375d59b40,3dc9bb2d-5d66-3e61-bf9a-e234c6433577,bb17e691-262b-3546-93d5-d88e7de93246,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,wellness,162673000,General examination of patient (procedure),...,10116,0,5465,0,8281,0,3644,99,1.134518,35196
4,dae2b7cb-1316-4b78-954f-fa610a6c6d0e,2010-07-27T12:58:08Z,2010-07-27T13:28:08Z,10339b10-3cd1-4ac3-ac13-ec26728cb592,b03dba4f-892f-365c-bfd1-bfcfa7a98d5d,7ed6b84a-b847-3744-9d42-15c42297a0c2,d47b3510-2895-3b70-9897-342d681c769d,wellness,162673000,General examination of patient (procedure),...,1796,0,4985,12718,550,6832,195,281,0.911843,70308


In [18]:
payer_billing_summary = encounters_denorm.groupby('NAME_payer').agg(
    encounter_count=('Id', 'count'),
    average_encounter_cost=('TOTAL_CLAIM_COST', 'mean'),
    total_billing=('TOTAL_CLAIM_COST', 'sum'),
    max_encounter_cost=('TOTAL_CLAIM_COST', 'max')
).reset_index().sort_values(by='average_encounter_cost', ascending=False)

display(payer_billing_summary)

,NAME_payer,encounter_count,average_encounter_cost,total_billing,max_encounter_cost
1,Anthem,2273,129.160000,293580.68,129.16
7,Medicare,9645,129.149286,1245644.86,129.16
0,Aetna,3433,128.994439,442837.91,129.16
3,Cigna Health,3766,128.954198,485641.51,129.16
4,Dual Eligible,715,128.943203,92194.39,129.16
2,Blue Cross Blue Shield,6604,128.815742,850699.16,129.16
8,NO_INSURANCE,10175,128.789296,1310431.09,129.16
9,UnitedHealthcare,4654,128.627091,598630.48,129.16
6,Medicaid,7966,128.271375,1021809.77,129.16
5,Humana,4115,128.029915,526843.10,129.16


In [19]:
procedures = pd.read_csv(f'{csv_dir}/procedures.csv')

inpatient_encounters = encounters_denorm[encounters_denorm['ENCOUNTERCLASS'] == 'inpatient']

inpatient_procedures = procedures.merge(inpatient_encounters, left_on='ENCOUNTER', right_on='Id', suffixes=('_proc', '_enc'))

top_inpatient_procedures = inpatient_procedures.groupby('DESCRIPTION_proc').agg(
    procedure_count=('CODE_proc', 'count'),
    avg_procedure_base_cost=('BASE_COST', 'mean')
).reset_index().sort_values(by='procedure_count', ascending=False).head(5)

display(top_inpatient_procedures)

,DESCRIPTION_proc,procedure_count,avg_procedure_base_cost
4,Combined chemotherapy and radiation therapy (p...,572,14291.907063
13,High resolution computed tomography of chest w...,328,516.650000
12,Hearing examination (procedure),328,516.650000
21,Mammography (procedure),135,516.650000
6,Cytopathology procedure preparation of smear ...,119,2472.467899


In [20]:
allergies = pd.read_csv(f'{csv_dir}/allergies.csv')

patient_allergies = allergies.merge(patients, left_on='PATIENT', right_on='Id')

allergy_geo_distribution = patient_allergies.groupby(['CITY', 'DESCRIPTION']).agg(
    allergy_count=('PATIENT', 'count')
).reset_index().sort_values(by='allergy_count', ascending=False).head(10)

display(allergy_geo_distribution)

,CITY,DESCRIPTION,allergy_count
55,Boston,Allergy to mould,9
61,Boston,House dust mite allergy,9
60,Boston,Dander (animal) allergy,9
54,Boston,Allergy to grass pollen,8
58,Boston,Allergy to tree pollen,8
62,Boston,Latex allergy,4
59,Boston,Allergy to wheat,4
63,Boston,Shellfish allergy,3
317,Quincy,Allergy to tree pollen,3
52,Boston,Allergy to eggs,3


In [21]:
medications = pd.read_csv(f'{csv_dir}/medications.csv')

emergency_encounters = encounters_denorm[encounters_denorm['ENCOUNTERCLASS'] == 'emergency']

emergency_meds = medications.merge(emergency_encounters, left_on='ENCOUNTER', right_on='Id', suffixes=('_med', '_enc'))

top_emergency_meds = emergency_meds.groupby('DESCRIPTION_med').agg(
    prescription_count=('CODE_med', 'count')
).reset_index().sort_values(by='prescription_count', ascending=False).head(10)

display(top_emergency_meds)

,DESCRIPTION_med,prescription_count
9,Acetaminophen 325 MG Oral Tablet,192
26,Ibuprofen 200 MG Oral Tablet,182
29,Naproxen sodium 220 MG Oral Tablet,166
6,Acetaminophen 160 MG Chewable Tablet,65
25,Ibuprofen 100 MG Oral Tablet,51
27,Meperidine Hydrochloride 50 MG Oral Tablet,35
10,Alteplase 100 MG Injection,27
8,Acetaminophen 325 MG / oxyCODONE Hydrochloride...,24
20,Clopidogrel 75 MG Oral Tablet,24
7,Acetaminophen 325 MG / HYDROcodone Bitartrate ...,21
